# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a complete workflow for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

#### Dataset Source
The dataset source is provided via a Croissant schema URL:
- `https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and inspect the dataset using `mlcroissant`. This includes basic information such as name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as object attributes
print("Dataset loaded!")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished if hasattr(dataset.metadata, 'datePublished') else 'N/A'}")

## 2. Data Overview

Explore available record sets and their fields, referencing all IDs using their `@id` as required by the Croissant standard.

In [ ]:
# List the available record sets, their @id, and field/column information
record_sets = dataset.metadata.record_sets

print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet name: {rs.name}")
    print(f"    @id: {rs.id}")
    record_set_ids.append(rs.id)
    print(f"    Description: {rs.description if hasattr(rs, 'description') else 'None'}")
    if hasattr(rs, 'fields'):
        print("    Fields (with `@id`):")
        for f in rs.fields:
            print(f"        - {f.name} (@id: {f.id}, dataType: {f.data_type if hasattr(f, 'data_type') else 'N/A'})")
    print()

# Save one as example for the next section
if len(record_set_ids) > 0:
    example_record_set_id = record_set_ids[0]
else:
    example_record_set_id = None

## 3. Data Extraction

Load records from the main record set(s) using their `@id`. Data is loaded into Pandas DataFrames. You can inspect the first few records and column names.

In [ ]:
# Build DataFrames for all record sets. Reference only by @id.
dataframes = {}
selected_record_set_id = example_record_set_id  # or change to another as needed

for rec_id in record_set_ids:
    print(f"Loading records from record set @id={rec_id}...")
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records.")

if selected_record_set_id is not None:
    print("\nColumn names of selected record set:")
    print(dataframes[selected_record_set_id].columns.tolist())
    print("\nFirst five rows:")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (e.g., a GAD-7, PHQ-9, or PSQ score) by its `@id` and analyze/filter/normalize.

Replace the selected field `@id` and threshold as appropriate for the dataset.

In [ ]:
import numpy as np

# Choose a numeric field by @id. Update this with the actual field id as found above, for example:
numeric_field = None
group_field = None

# Try to automatically identify a common anxiety or symptom score field
possible_numeric_fields = [c for c in dataframes[selected_record_set_id].columns if any(x in c.lower() for x in ['score', 'phq', 'gad', 'psq', 'numeric', 'age'])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Use the first

# Optionally group by another field (e.g. gender, education_level)
possible_group_fields = [c for c in dataframes[selected_record_set_id].columns if any(x in c.lower() for x in ['gender', 'sex', 'education', 'marital', 'village'])]
if possible_group_fields:
    group_field = possible_group_fields[0]

if numeric_field is None:
    print("No obvious numeric field was found. Please examine the DataFrame columns and set `numeric_field` and `group_field`.")
else:
    print(f"Analyzing numeric field '{numeric_field}' (from @id)")
    # Remove na and non-numeric
    df = dataframes[selected_record_set_id].copy()

    # Try to convert to float
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = np.nanmedian(df[numeric_field])

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.1f} (median): {len(filtered_df)} rows out of {len(df)}")
    display(filtered_df.head())

    # Normalize values
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Showing normalized '{numeric_field}':")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None and group_field in filtered_df.columns:
        print(f"\nGrouping by {group_field} and computing group means:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename(columns={numeric_field: f"mean_{numeric_field}"})
        display(grouped_df)
    else:
        print("No group field available.")

## 5. Visualization

Visualize the distribution of the selected numeric field. We'll use a histogram and boxplot, and if a group field is present, show group means.

**All references to columns are by their `@id`.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is None:
    print("Please define `numeric_field` above to visualize.")
else:
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    sns.histplot(data=dataframes[selected_record_set_id], x=numeric_field, bins=15, kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1, 2, 2)
    sns.boxplot(data=dataframes[selected_record_set_id], x=numeric_field, color='orange')
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

    if group_field is not None and group_field in dataframes[selected_record_set_id].columns:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=dataframes[selected_record_set_id], x=group_field, y=numeric_field, ci=None)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.
- We loaded metadata and records, referenced all fields and sets by their Croissant `@id`, and performed simple filtering, normalization, and visualization steps on the data.
- Adapt the field selection and analysis to your project needs—each field and record set is accessible via the Croissant `@id` system for reproducibility and clarity.

For further analysis, consult the [dataset documentation](https://sen.science/doi/10.71728/senscience.vcs2-05nj) or the Croissant schema file.